In [ ]:
# ============================================================
# RailGuard — YOLOv8
# Testing & Error Analysis
# ============================================================

!pip install -q ultralytics==8.4.172

import os
import time
import pandas as pd
import numpy as np

from pathlib import Path
from ultralytics import YOLO

import ultralytics

print("Ultralytics:", ultralytics.__version__)

In [ ]:
# ============================================================
# TEST CONFIGURATION
# ============================================================

MODEL_PATH = Path(
    "/kaggle/working/"
    "railguard_yolov8/"
    "training/"
    "weights/"
    "best.pt"
)

DATASET_ROOT = Path(
    "/kaggle/working/"
    "railway_defect_binary"
)

DATA_YAML = (
    DATASET_ROOT /
    "data.yaml"
)

TEST_IMAGES = (
    DATASET_ROOT /
    "test" /
    "images"
)

TEST_LABELS = (
    DATASET_ROOT /
    "test" /
    "labels"
)

RESULTS_ROOT = Path(
    "/kaggle/working/"
    "railguard_yolov8"
)

TEST_EVAL_DIR = (
    RESULTS_ROOT /
    "test_evaluation"
)

PREDICTION_DIR = (
    RESULTS_ROOT /
    "test_predictions"
)

ERROR_DIR = (
    RESULTS_ROOT /
    "error_examples"
)

print("Model:", MODEL_PATH)
print("Dataset:", DATASET_ROOT)
print("Test images:", TEST_IMAGES)
print("Test labels:", TEST_LABELS)

In [ ]:
# ============================================================
# VERIFY TESTING FILES
# ============================================================

print("Model exists:", MODEL_PATH.exists())
print("Dataset exists:", DATASET_ROOT.exists())
print("YAML exists:", DATA_YAML.exists())
print("Test images:", TEST_IMAGES.exists())
print("Test labels:", TEST_LABELS.exists())

if TEST_IMAGES.exists():

    test_images = [
        p for p in TEST_IMAGES.iterdir()
        if p.is_file()
    ]

    print(
        "\nNumber of test images:",
        len(test_images)
    )

if TEST_LABELS.exists():

    test_labels = list(
        TEST_LABELS.glob("*.txt")
    )

    print(
        "Number of label files:",
        len(test_labels)
    )

In [ ]:
# ============================================================
# TEST DATASET STATISTICS
# ============================================================

test_images = [
    p for p in TEST_IMAGES.iterdir()
    if p.is_file()
]

test_labels = list(
    TEST_LABELS.glob("*.txt")
)

total_objects = 0
empty_labels = 0

class_ids = {}

for label in test_labels:

    lines = [
        line.strip()
        for line in label.read_text().splitlines()
        if line.strip()
    ]

    if not lines:
        empty_labels += 1

    for line in lines:

        parts = line.split()

        if len(parts) >= 5:

            class_id = int(parts[0])

            class_ids[class_id] = (
                class_ids.get(class_id, 0) + 1
            )

            total_objects += 1

print("=" * 50)
print("TEST DATASET")
print("=" * 50)

print("Images:", len(test_images))
print("Label files:", len(test_labels))
print("Objects:", total_objects)
print("Empty labels:", empty_labels)

print("\nClass IDs:")
print(sorted(class_ids.keys()))

print("\nClass distribution:")
print(class_ids)

In [ ]:
# ============================================================
# LOAD TRAINED YOLOv8 MODEL
# ============================================================

model = YOLO(
    str(MODEL_PATH)
)

print("Model loaded successfully.")
print("Model:", MODEL_PATH)

print("\nModel classes:")
print(model.names)

In [ ]:
# ============================================================
# FINAL TEST-SET EVALUATION
# ============================================================

start_time = time.time()

test_results = model.val(

    data=str(DATA_YAML),

    split="test",

    imgsz=640,
    batch=16,

    device=0,

    plots=True,

    project=str(RESULTS_ROOT),
    name="test_evaluation",

    exist_ok=True
)

evaluation_time = (
    time.time() -
    start_time
)

print("\nTest evaluation completed.")

print(
    f"Evaluation time: "
    f"{evaluation_time:.2f} seconds"
)

In [ ]:
# ============================================================
# TEST METRICS
# ============================================================

metrics = (
    test_results.results_dict
)

precision = float(
    metrics.get(
        "metrics/precision(B)",
        0
    )
)

recall = float(
    metrics.get(
        "metrics/recall(B)",
        0
    )
)

map50 = float(
    metrics.get(
        "metrics/mAP50(B)",
        0
    )
)

map5095 = float(
    metrics.get(
        "metrics/mAP50-95(B)",
        0
    )
)

f1 = (
    2 * precision * recall /
    (precision + recall)
    if precision + recall > 0
    else 0
)

print("=" * 55)
print("YOLOv8 FINAL TEST RESULTS")
print("=" * 55)

print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"mAP@50       : {map50:.4f}")
print(f"mAP@50-95    : {map5095:.4f}")
print(f"Evaluation   : {evaluation_time:.2f} seconds")

In [ ]:
# ============================================================
# SAVE FINAL TEST EVALUATION
# ============================================================

test_log = pd.DataFrame({

    "model": ["YOLOv8n"],

    "dataset": [
        "Mendeley Railway Defects Detection"
    ],

    "task": [
        "Binary Railway Defect Detection"
    ],

    "split": ["test"],

    "images": [310],

    "instances": [1494],

    "precision": [precision],

    "recall": [recall],

    "f1_score": [f1],

    "mAP50": [map50],

    "mAP50_95": [map5095],

    "evaluation_time_seconds": [
        evaluation_time
    ]
})

TEST_LOG = (
    TEST_EVAL_DIR /
    "test_evaluation.csv"
)

test_log.to_csv(
    TEST_LOG,
    index=False
)

print("Saved:")
print(TEST_LOG)

In [ ]:
# ============================================================
# TEST SET PREDICTIONS
# ============================================================

start_time = time.time()

prediction_results = model.predict(

    source=str(TEST_IMAGES),

    imgsz=640,

    conf=0.25,

    device=0,

    save=True,
    save_txt=True,
    save_conf=True,

    project=str(RESULTS_ROOT),
    name="test_predictions",

    exist_ok=True,

    verbose=False
)

prediction_time = (
    time.time() -
    start_time
)

print("=" * 55)
print("YOLOv8 PREDICTION COMPLETE")
print("=" * 55)

print(
    "Images processed:",
    len(prediction_results)
)

print(
    f"Total time: "
    f"{prediction_time:.2f} seconds"
)

print(
    f"Average time/image: "
    f"{prediction_time / len(prediction_results) * 1000:.2f} ms"
)

print(
    "Prediction directory:",
    PREDICTION_DIR
)

prediction_labels = (
    PREDICTION_DIR /
    "labels"
)

if prediction_labels.exists():

    print(
        "Prediction label files:",
        len(
            list(
                prediction_labels.glob("*.txt")
            )
        )
    )

In [ ]:
# ============================================================
# CONFIDENCE 0.50 PREDICTIONS
# ============================================================

CONF50_DIR = (
    RESULTS_ROOT /
    "test_predictions_conf50"
)

confidence_results = model.predict(

    source=str(TEST_IMAGES),

    imgsz=640,

    conf=0.50,

    device=0,

    save=True,
    save_txt=True,
    save_conf=True,

    project=str(RESULTS_ROOT),
    name="test_predictions_conf50",

    exist_ok=True,

    verbose=False
)

print("Confidence 0.50 prediction run completed.")

print(
    "Output:",
    CONF50_DIR
)

In [ ]:
# ============================================================
# FP/FN ERROR ANALYSIS
#
# Matching rule:
# - Ground-truth and prediction boxes are matched using IoU.
# - A prediction is TP when its best unmatched ground-truth
#   box reaches the IoU threshold.
# - Unmatched predictions = FP.
# - Unmatched ground-truth boxes = FN.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

IOU_THRESHOLD = 0.50

GT_DIR = TEST_LABELS
PRED_DIR = PREDICTION_DIR / "labels"


def read_yolo_boxes(label_path):

    boxes = []

    if not label_path.exists():
        return boxes

    for line in label_path.read_text().splitlines():

        parts = line.split()

        if len(parts) < 5:
            continue

        _, x, y, w, h = map(
            float,
            parts[:5]
        )

        x1 = x - w / 2
        y1 = y - h / 2
        x2 = x + w / 2
        y2 = y + h / 2

        boxes.append(
            [x1, y1, x2, y2]
        )

    return boxes


def calculate_iou(box_a, box_b):

    xa1 = max(box_a[0], box_b[0])
    ya1 = max(box_a[1], box_b[1])

    xa2 = min(box_a[2], box_b[2])
    ya2 = min(box_a[3], box_b[3])

    intersection_w = max(
        0,
        xa2 - xa1
    )

    intersection_h = max(
        0,
        ya2 - ya1
    )

    intersection = (
        intersection_w *
        intersection_h
    )

    area_a = (
        max(0, box_a[2] - box_a[0]) *
        max(0, box_a[3] - box_a[1])
    )

    area_b = (
        max(0, box_b[2] - box_b[0]) *
        max(0, box_b[3] - box_b[1])
    )

    union = (
        area_a +
        area_b -
        intersection
    )

    if union <= 0:
        return 0.0

    return intersection / union


image_results = []

total_tp = 0
total_fp = 0
total_fn = 0

for image in sorted(TEST_IMAGES.iterdir()):

    if not image.is_file():
        continue

    gt_file = (
        GT_DIR /
        f"{image.stem}.txt"
    )

    pred_file = (
        PRED_DIR /
        f"{image.stem}.txt"
    )

    gt_boxes = read_yolo_boxes(
        gt_file
    )

    pred_boxes = read_yolo_boxes(
        pred_file
    )

    matched_gt = set()

    tp = 0
    fp = 0

    for pred in pred_boxes:

        best_iou = 0
        best_gt = None

        for index, gt in enumerate(gt_boxes):

            if index in matched_gt:
                continue

            iou = calculate_iou(
                pred,
                gt
            )

            if iou > best_iou:

                best_iou = iou
                best_gt = index

        if (
            best_gt is not None
            and best_iou >= IOU_THRESHOLD
        ):

            tp += 1
            matched_gt.add(best_gt)

        else:
            fp += 1

    fn = (
        len(gt_boxes) -
        len(matched_gt)
    )

    total_tp += tp
    total_fp += fp
    total_fn += fn

    if tp > 0 and fp > 0 and fn > 0:
        category = "TP+FP+FN"

    elif fp > 0 and fn > 0:
        category = "FP+FN"

    elif fp > 0:
        category = "FP"

    elif fn > 0:
        category = "FN"

    elif tp > 0:
        category = "TP"

    else:
        category = "TN"

    image_results.append({

        "image": image.name,

        "ground_truth_objects":
            len(gt_boxes),

        "predicted_objects":
            len(pred_boxes),

        "TP": tp,

        "FP": fp,

        "FN": fn,

        "error_type": category
    })


error_df = pd.DataFrame(
    image_results
)

ERROR_CSV = (
    RESULTS_ROOT /
    "error_analysis.csv"
)

error_df.to_csv(
    ERROR_CSV,
    index=False
)

print("=" * 55)
print("YOLOv8 ERROR ANALYSIS")
print("=" * 55)

print(
    "Images analyzed:",
    len(error_df)
)

print(
    "Ground-truth objects:",
    total_tp + total_fn
)

print(
    "Predicted objects:",
    total_tp + total_fp
)

print("TP:", total_tp)
print("FP:", total_fp)
print("FN:", total_fn)

print("\nImage-level categories:")

print(
    error_df[
        "error_type"
    ].value_counts()
)

print("\nSaved:")
print(ERROR_CSV)

In [ ]:
# ============================================================
# GENERATE FALSE-POSITIVE / FALSE-NEGATIVE EXAMPLES
# ============================================================

import shutil

FP_DIR = (
    ERROR_DIR /
    "false_positives"
)

FN_DIR = (
    ERROR_DIR /
    "false_negatives"
)

FP_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

fp_images = error_df[
    error_df["FP"] > 0
].sort_values(
    "FP",
    ascending=False
)

fn_images = error_df[
    error_df["FN"] > 0
].sort_values(
    "FN",
    ascending=False
)

fp_selected = fp_images.head(10)
fn_selected = fn_images.head(10)

for _, row in fp_selected.iterrows():

    source = (
        TEST_IMAGES /
        row["image"]
    )

    if source.exists():

        shutil.copy2(
            source,
            FP_DIR /
            row["image"]
        )


for _, row in fn_selected.iterrows():

    source = (
        TEST_IMAGES /
        row["image"]
    )

    if source.exists():

        shutil.copy2(
            source,
            FN_DIR /
            row["image"]
        )

print("============================================")
print("YOLOv8 ERROR EXAMPLES GENERATED")
print("============================================")

print(
    "False-positive examples:",
    len(list(FP_DIR.iterdir()))
)

print(
    "False-negative examples:",
    len(list(FN_DIR.iterdir()))
)

print("\nFP directory:")
print(FP_DIR)

print("\nFN directory:")
print(FN_DIR)

In [ ]:
# ============================================================
# VERIFY ERROR ANALYSIS ARTIFACTS
# ============================================================

print("=" * 55)
print("ERROR ANALYSIS ARTIFACT VERIFICATION")
print("=" * 55)

print(
    "error_analysis.csv:",
    ERROR_CSV.exists()
)

print(
    "FP directory:",
    FP_DIR.exists()
)

print(
    "FN directory:",
    FN_DIR.exists()
)

if FP_DIR.exists():

    print(
        "FP examples:",
        len(list(FP_DIR.iterdir()))
    )

if FN_DIR.exists():

    print(
        "FN examples:",
        len(list(FN_DIR.iterdir()))
    )

In [ ]:
# ============================================================
# FINAL TESTING SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("RAILGUARD — YOLOv8 FINAL TEST REPORT")
print("=" * 65)

print("\nMODEL")
print("-" * 40)
print("Architecture : YOLOv8n")
print("Task         : Binary Railway Defect Detection")
print("Class        : defect")

print("\nTEST DATA")
print("-" * 40)
print("Images       : 310")
print("Instances    : 1494")

print("\nTEST METRICS")
print("-" * 40)
print(f"Precision    : {precision:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"F1-score     : {f1:.4f}")
print(f"mAP@50       : {map50:.4f}")
print(f"mAP@50-95    : {map5095:.4f}")

print("\nERROR ANALYSIS")
print("-" * 40)
print("TP           :", total_tp)
print("FP           :", total_fp)
print("FN           :", total_fn)

print("\nPREDICTION SPEED")
print("-" * 40)
print(
    f"Average      : "
    f"{prediction_time / len(prediction_results) * 1000:.2f} ms/image"
)

print("\nARTIFACTS")
print("-" * 40)
print("Test evaluation :", TEST_EVAL_DIR)
print("Predictions     :", PREDICTION_DIR)
print("Error analysis  :", ERROR_CSV)
print("FP examples     :", FP_DIR)
print("FN examples     :", FN_DIR)

print("\nYOLOv8 testing completed successfully.")